# 03 — Preprocessing

The purpose of this notebook is to prepare the dataset for machine learning modeling.

In this notebook, we will:
- Load the raw data
- Remove identifier columns
- Separate features and target
- Split the data into training and test sets
- Identify numerical and categorical variables
- Build a preprocessing pipeline
- Transform the data into a model-ready format
- Save the preprocessing objects and transformed datasets for the modeling notebook

In [40]:
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [41]:
# Load the dataset
df = pd.read_csv('../data/raw/Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 1. Drop identifier columns

Some columns uniquely identify customers but are not useful for prediction.

In [42]:
cols_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df_model = df.drop(columns=cols_to_drop).copy()
df_model.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


``RowNumber``, ``CustomerId``, and ``Surname`` were removed because they do not represent meaningful behavioral or financial information about the customer. They are mainly identifiers, so they are unlikely to help the model predict churn and may instead introduce unnecessary noise.

## 2. Separate features and target

`Exited` is the target variable. All other columns are candidate features.

In [43]:
X = df_model.drop(columns='Exited')
y = df_model['Exited']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nTarget distribution (%):')
print(y.value_counts(normalize=True).mul(100).round(2))

X shape: (10000, 10)
y shape: (10000,)

Target distribution (%):
Exited
0   79.63
1   20.37
Name: proportion, dtype: float64


With this we know the X and Y shapes and the target distribution

## 3. Identify numerical and categorical columns

This helps us decide which preprocessing steps to apply to each group.

In [44]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

print('Categorical columns:', cat_cols)
print('Numerical columns:', num_cols)

Categorical columns: ['Geography', 'Gender']
Numerical columns: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


Categorical columns need to be encoded because machine learning models generally require numerical input. Features such as ``Geography`` and ``Gender`` contain text values, so they must be transformed into numerical form before training the model.

## 4. Train-test split

We use a stratified split to preserve the class distribution of the target variable.

In [45]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('\nTrain target distribution (%):')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print('\nTest target distribution (%):')
print(y_test.value_counts(normalize=True).mul(100).round(2))

X_train shape: (8000, 10)
X_test shape: (2000, 10)

Train target distribution (%):
Exited
0   79.62
1   20.38
Name: proportion, dtype: float64

Test target distribution (%):
Exited
0   79.65
1   20.35
Name: proportion, dtype: float64


## 5. Build the preprocessing pipeline

We will:
- scale numerical columns
- one-hot encode categorical columns

This is a common and clean approach for preparing structured tabular data.

In [46]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['CreditScore', 'Age', 'Tenure', 'Balance',
                                  'NumOfProducts', 'HasCrCard',
                                  'IsActiveMember', 'EstimatedSalary']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Geography', 'Gender'])])

Scaling is applied to numerical features so that variables with very different ranges can be placed on a comparable scale. This helps many machine learning algorithms learn more effectively and prevents variables with larger values from having an excessive influence on the model.

## 6. Fit the preprocessor on the training data only

This is important to avoid data leakage.

In [47]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Processed X_train shape:', X_train_processed.shape)
print('Processed X_test shape:', X_test_processed.shape)

Processed X_train shape: (8000, 11)
Processed X_test shape: (2000, 11)


The preprocessor is fitted only on the training set to prevent data leakage. This ensures that information from the test set does not influence preprocessing decisions such as scaling or encoding, which makes model evaluation more realistic and trustworthy.

## 7. Recover transformed feature names

This makes the processed data easier to inspect and save as dataframes.

In [48]:
encoded_cat_cols = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols).tolist()
processed_columns = num_cols + encoded_cat_cols

print('Number of processed columns:', len(processed_columns))
print(processed_columns)

Number of processed columns: 11
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']


In [49]:
X_train_processed_df = pd.DataFrame(X_train_processed, columns=processed_columns, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=processed_columns, index=X_test.index)

X_train_processed_df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2151,1.06,1.72,0.68,-1.23,-0.91,0.64,-1.03,1.04,0.00,0.00,1.00
8392,0.91,-0.66,-0.70,0.41,-0.91,0.64,-1.03,-0.62,1.00,0.00,1.00
5006,1.08,-0.18,-1.73,0.60,0.81,0.64,0.97,0.31,1.00,0.00,0.00
4117,-0.93,-0.18,-0.01,-1.23,0.81,0.64,-1.03,-0.29,0.00,0.00,1.00
7182,0.43,0.96,0.34,0.55,0.81,-1.56,0.97,0.14,1.00,0.00,1.00


## 8. Save processed datasets

We save the processed train and test sets so the next notebook can focus on modeling.

In [50]:
X_train_processed_df.to_csv('../data/processed/X_train_processed.csv', index=False)
X_test_processed_df.to_csv('../data/processed/X_test_processed.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.joblib')

print('Processed files and preprocessor saved successfully.')

Processed files and preprocessor saved successfully.


## 9. Quick verification

A final quick check before moving to modeling.

In [51]:
print('Missing values in processed X_train:', X_train_processed_df.isnull().sum().sum())
print('Missing values in processed X_test:', X_test_processed_df.isnull().sum().sum())
print('Train rows:', X_train_processed_df.shape[0], '| Test rows:', X_test_processed_df.shape[0])

Missing values in processed X_train: 0
Missing values in processed X_test: 0
Train rows: 8000 | Test rows: 2000
